In [1]:
import pandas as pd
import numpy as np
import joblib
from scipy import sparse

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Load artifacts from Notebook 5
X_train_processed = sparse.load_npz("X_train_processed.npz")
X_val_processed = sparse.load_npz("X_val_processed.npz")
X_test_processed = sparse.load_npz("X_test_processed.npz")

y_train = pd.read_csv("y_train.csv").squeeze("columns")
y_val = pd.read_csv("y_val.csv").squeeze("columns")
y_test = pd.read_csv("y_test.csv").squeeze("columns")

preprocessor = joblib.load("preprocessor.pkl")

print("Artifacts loaded successfully!")
print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

Artifacts loaded successfully!
Train: (67533, 3789)
Validation: (14471, 3789)
Test: (14472, 3789)


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

log_model = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("logreg", LogisticRegression(
        solver="liblinear",
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

log_model.fit(X_train_processed, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [3]:
log_pred = log_model.predict(X_val_processed)
log_prob = log_model.predict_proba(X_val_processed)[:, 1]

print("Accuracy :", accuracy_score(y_val, log_pred))
print("Precision:", precision_score(y_val, log_pred))
print("Recall   :", recall_score(y_val, log_pred))
print("F1 Score :", f1_score(y_val, log_pred))
print("ROC-AUC  :", roc_auc_score(y_val, log_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, log_pred))

Accuracy : 0.6821919701471909
Precision: 0.1017901748542881
Recall   : 0.6326002587322122
F1 Score : 0.17536309844002151
ROC-AUC  : 0.7027442557312358

Confusion Matrix:
[[9383 4315]
 [ 284  489]]


In [4]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_processed, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [5]:
rf_pred = rf_model.predict(X_val_processed)
rf_prob = rf_model.predict_proba(X_val_processed)[:, 1]

print("Random Forest - Validation Results")
print("Accuracy :", accuracy_score(y_val, rf_pred))
print("Precision:", precision_score(y_val, rf_pred))
print("Recall   :", recall_score(y_val, rf_pred))
print("F1 Score :", f1_score(y_val, rf_pred))
print("ROC-AUC  :", roc_auc_score(y_val, rf_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, rf_pred))

Random Forest - Validation Results
Accuracy : 0.6259415382489116
Precision: 0.08288385472851492
Recall   : 0.5963777490297542
F1 Score : 0.14554064719810575
ROC-AUC  : 0.6603481457430355

Confusion Matrix:
[[8597 5101]
 [ 312  461]]


In [6]:
thresholds = np.arange(0.10, 0.95, 0.05)

threshold_results = []

for threshold in thresholds:
    preds = (log_prob >= threshold).astype(int)

    precision = precision_score(y_val, preds, zero_division=0)
    recall = recall_score(y_val, preds)
    f1 = f1_score(y_val, preds)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results = pd.DataFrame(threshold_results)

threshold_results

,threshold,precision,recall,f1
0,0.10,0.055653,0.906856,0.104869
1,0.15,0.057304,0.906856,0.107796
2,0.20,0.059800,0.904269,0.112181
3,0.25,0.063292,0.891332,0.118192
4,0.30,0.067462,0.870634,0.125221
5,0.35,0.073129,0.834411,0.134473
6,0.40,0.080972,0.780078,0.146715
7,0.45,0.090820,0.721863,0.161342
8,0.50,0.101790,0.632600,0.175363
9,0.55,0.114992,0.553687,0.190434


In [7]:
best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = best_row["threshold"]

print("Best threshold:", best_threshold)
print("Validation Precision:", best_row["precision"])
print("Validation Recall:", best_row["recall"])
print("Validation F1:", best_row["f1"])

Best threshold: 0.7500000000000002
Validation Precision: 0.18090452261306533
Validation Recall: 0.23285899094437257
Validation F1: 0.20361990950226244


In [8]:
# Final evaluation on TEST set

test_prob = log_model.predict_proba(X_test_processed)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)

test_accuracy = accuracy_score(y_test, test_pred)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred)
test_f1 = f1_score(y_test, test_pred)
test_auc = roc_auc_score(y_test, test_prob)

print("Final Test Results")
print("Accuracy :", test_accuracy)
print("Precision:", test_precision)
print("Recall   :", test_recall)
print("F1 Score :", test_f1)
print("ROC-AUC  :", test_auc)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

Final Test Results
Accuracy : 0.8408651188501934
Precision: 0.1003562945368171
Recall   : 0.17659352142110762
F1 Score : 0.12798182506626277
ROC-AUC  : 0.6314863588620716

Confusion Matrix:
[[12000  1515]
 [  788   169]]


In [9]:
import joblib
import json

# Save trained model and preprocessor
joblib.dump(log_model, "logistic_regression_model.pkl")


# Save final results
results = {
    "model": "Logistic Regression",
    "best_threshold": float(best_threshold),
    "test_accuracy": float(test_accuracy),
    "test_precision": float(test_precision),
    "test_recall": float(test_recall),
    "test_f1": float(test_f1),
    "test_roc_auc": float(test_auc)
}

with open("results_summary.json", "w") as f:
    json.dump(results, f, indent=4)

print("Model saved successfully!")
print("Results saved successfully!")

Model saved successfully!
Results saved successfully!
